# Case 03 · QLoRA Fine-Tuning a Real LLM

**What you’ll do:** Load a real LLM in 4-bit, attach LoRA adapters, fine-tune on an instruction dataset, compare before/after, and save the adapter.

**Runs on:** Google Colab free T4 (16 GB) — or any NVIDIA GPU with ≥12 GB VRAM.

| Step | What happens |
|------|-------------|
| 1 | Install + import |
| 2 | Load model in 4-bit (BitsAndBytesConfig) |
| 3 | Attach LoRA (peft) |
| 4 | Load + format dataset |
| 5 | Generate BEFORE training |
| 6 | Train with SFTTrainer |
| 7 | Generate AFTER training |
| 8 | Save the adapter |
| 9 | (Optional) Merge into one model |

> **Default model:** `Qwen/Qwen2.5-0.5B-Instruct` — ungated, tiny, runs in minutes.  
> **Swap to Llama-3-8B** when you’re ready — just change `MODEL_ID` below (and see `GET_LLAMA3.md` for access).

## 0 · Setup

Run this cell once. On Colab it installs everything; locally it’s a no-op if your venv already has the packages.

In [ ]:
# Install (Colab: ~2 min; skip if already installed locally)
!pip install -q torch transformers accelerate peft bitsandbytes trl datasets

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

print(f"torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1 · Configuration

All the knobs in one place. Change `MODEL_ID` to try a bigger model.

In [ ]:
# ---- Model ----
# Starter: ungated, tiny, fast (no license needed)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# When you're ready for a real LLM (needs HF login + license, see GET_LLAMA3.md):
# MODEL_ID = "meta-llama/Meta-Llama-3-8B"

# ---- LoRA ----
LORA_R       = 8       # rank: 4-64 typical; lower = cheaper
LORA_ALPHA   = 16      # scaling; common default = 2*r
LORA_DROPOUT = 0.05

# ---- Training ----
EPOCHS        = 1
BATCH_SIZE    = 4
GRAD_ACCUM    = 2      # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 512
NUM_EXAMPLES  = 1000   # subset of the dataset (keep small for learning)

OUTPUT_DIR = "./qlora-adapter"

## 2 · Load the model in 4-bit

This is the **Q** in QLoRA. `BitsAndBytesConfig` tells HuggingFace to quantize every weight to 4-bit NF4 as it loads. The model lands on the GPU already compressed.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normal Float 4 — best for weights
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16 for speed + accuracy
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"\nModel loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"dtype of first linear: {next(model.parameters()).dtype}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3 · Attach LoRA adapters

The base model is frozen in 4-bit. Now we bolt on tiny trainable adapters (the **LoRA** from Case 02, but using the `peft` library instead of our hand-rolled version).

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",  # attach to every linear layer
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4 · Load and format the dataset

We use a 1K subset of the [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca) instruction dataset. Each example has an instruction, optional input, and expected output.

We format them into a single text string that the model can learn to complete.

In [ ]:
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(NUM_EXAMPLES))

def format_example(example):
    """Format as Alpaca-style prompt + completion."""
    text = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get("input", "").strip():
        text += f"### Input:\n{example['input']}\n\n"
    text += f"### Response:\n{example['output']}"
    return {"text": text}

dataset = dataset.map(format_example)

print(f"Dataset: {len(dataset)} examples")
print(f"\n--- Example ---\n{dataset[0]['text'][:400]}...")

## 5 · Generate BEFORE training

Let’s see what the model does on a few prompts *before* we fine-tune. We’ll compare this to the output *after* training.

In [ ]:
test_prompts = [
    "### Instruction:\nExplain what QLoRA is in one sentence.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function that reverses a string.\n\n### Response:\n",
    "### Instruction:\nWhat are three benefits of fine-tuning a language model?\n\n### Response:\n",
]

def generate(model, tokenizer, prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("BEFORE TRAINING")
print("=" * 60)
before_outputs = []
for p in test_prompts:
    out = generate(model, tokenizer, p)
    before_outputs.append(out)
    print(f"\nPrompt: {p.split(chr(10))[1]}")
    print(f"Output: {out[:200]}")
    print("-" * 40)

## 6 · Train

`SFTTrainer` from `trl` wraps HuggingFace’s `Trainer` with extras for supervised fine-tuning: it tokenizes on the fly, handles padding, and works with LoRA out of the box.

With 1K examples on Qwen-0.5B, this finishes in **~5 minutes** on a T4.  
On Llama-3-8B it takes **~20–30 minutes**.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="no",          # we save manually at the end
    report_to="none",            # no wandb/mlflow noise
    gradient_checkpointing=True, # trades compute for VRAM
    optim="paged_adamw_8bit",    # 8-bit optimizer — saves more VRAM
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
)

print(f"Training {trainer.get_num_trainable_parameters():,} params on {len(dataset)} examples...")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps: {len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}")
print()

trainer.train()

if torch.cuda.is_available():
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 7 · Generate AFTER training

Same prompts as before. Look for:
- Does the model follow the instruction format better?
- Are the answers more structured / on-topic?
- Did it learn the Alpaca response style?

In [ ]:
print("=" * 60)
print("AFTER TRAINING")
print("=" * 60)
for i, p in enumerate(test_prompts):
    out = generate(model, tokenizer, p)
    print(f"\nPrompt: {p.split(chr(10))[1]}")
    print(f"Before: {before_outputs[i][:150]}")
    print(f"After:  {out[:150]}")
    print("-" * 40)

## 8 · Save the adapter

Only the LoRA weights get saved — typically **10–50 MB** vs 1–16 GB for the full model. You can share just this adapter; anyone with the same base model can load it.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
adapter_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
)
print(f"Adapter saved to: {OUTPUT_DIR}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"\nFiles:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f:40s} {os.path.getsize(fpath)/1e6:.1f} MB")

### Loading the adapter later

To use this adapter in a new session, you load the base model and then attach the saved adapter on top:

```python
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base, "./qlora-adapter")
```

You can have multiple adapters for different tasks and swap them on the same base. This is the starting point for Case 05 (catastrophic forgetting) — one adapter per task, no interference.

## 9 · (Optional) Merge into one model

`merge_and_unload()` folds the adapter back into the base weights: `W ← W + B·A·(α/r)`. The result is an ordinary model with no adapter overhead at inference.

In [ ]:
merged = model.merge_and_unload()
print(f"Merged model type: {type(merged).__name__}")
print(f"Parameters: {merged.num_parameters():,}")
print("\nThe adapter is gone — the model is now a regular transformers model.")
print("You could save it with: merged.save_pretrained('./merged-model')")

---

## Takeaways

1. **4-bit quantization** shrinks the model ~4× with minimal quality loss. That’s how an 8B model fits on a free T4.
2. **LoRA adapters** mean you only train ~0.5% of params. The optimizer state (the real memory hog) stays tiny.
3. **QLoRA = quantize + LoRA.** Two independent tricks that stack: small base + small trainable part.
4. **Adapters are portable:** 10–50 MB files you can save, share, swap, and stack.
5. **Merging** folds the adapter back in for deployment — zero overhead at inference.

### What to try next
- Change `LORA_R` (rank) and re-train. Does r=2 still work? Does r=64 help or just waste VRAM?
- Swap `MODEL_ID` to `meta-llama/Meta-Llama-3-8B` (see `GET_LLAMA3.md`) and re-run.
- Try a different dataset — your own Q&A pairs, code instructions, anything.
- Open **`challenge.md`** for guided experiments.

### Coming up
- **Case 04 · Instruction SFT:** proper evaluation — not just eyeballing outputs.
- **Case 05 · Catastrophic forgetting:** what happens when you train on task B? Does task A break?
- **Case 07 · Capstone:** streaming self-driving updates — cost vs accuracy vs forgetting.